# Chunking

Splits the 6 `Knowledgebase/*.xlsx` article tables (produced by `files_text_extraction.ipynb`) into retrieval-sized chunks and writes `0. cleaned_data/chunks.parquet`.

Most articles are already small enough to embed as a single chunk (median ~460 chars). A handful of long articles (multi-paragraph provisions, e.g. SCL Article 85 on combining pensions) get split on their lettered sub-clauses (`a-`, `b-` / `أ-`, `ب-`) so no single chunk blows past ~3000 characters.

This was originally prototyped as a few cells at the end of `files_text_extraction.ipynb`; moved here since it's a distinct pipeline stage (chunking, not extraction) with its own inputs/outputs. Two real bugs were caught and fixed while separating it out - see the note above the chunking cell below.

In [ ]:
import pandas as pd
import re
from pathlib import Path

KB = Path(r"C:\Users\Mohannad Alsaleh\Desktop\Data Science\Data Science Projects\1st RAG project\0. cleaned_data\Knowledgebase")

FILES = {
    "scl_articles_ar.xlsx":          ("ssl",     "ar"),
    "scl_articles_en.xlsx":          ("ssl",     "en"),
    "worklaw_ar_articles.xlsx":      ("labor96", "ar"),
    "worklaw_articles_en.xlsx":      ("labor96", "en"),
    "worklaw_prof_ar_articles.xlsx": ("prof19",  "ar"),
    "worklaw_prof_articles_en.xlsx": ("prof19",  "en"),
}

## Size overview

Quick look at article-length distribution per file before deciding what needs splitting.

In [ ]:
for fname in FILES:
    L = pd.read_excel(KB / fname)["Content"].str.len()
    print(f"{fname:32} n={len(L):3} med={int(L.median()):5} p90={int(L.quantile(.9)):5} max={L.max():5} over3k={int((L > 3000).sum())}")

## Clause-level splitting for long articles

Two bugs surfaced when this was reviewed after moving out of the extraction notebook, both from the original version's `getattr(r, "_2")` / raw-letter chunk-id approach:

1. **EN split regex only matched `a.`/`a)`, not `a-`.** SCL EN articles use period style (`a. The...`), but the 1996 Work Law EN articles almost all use hyphen style (`a- The...`). The old regex silently left every long Work Law EN article as one oversized, unsplit chunk (e.g. Article 54 at 4650 chars). Fixed by matching `.`, `-`/en-dash, or `)` as the separator for both languages.
2. **Duplicate `chunk_id`s from nested lettering.** Several long articles (SCL Articles 64, 71, 76, 85 in both languages) restart the alphabet inside a nested sub-list - e.g. Article 85 has top-level `a.`...`g.`, and *inside* clause `d`'s own numbered sub-item there's a second, independent `a.`...`h.` list. The old id (`{law}_{lang}_{art}_{clause_letter}`) collided on every repeated letter (found 90+ colliding rows in the real data - `ssl_en_85_a` alone appeared twice). Fixed by keying the id on the split's *position* within the article instead of the raw letter; the letter itself is kept as a `clause` metadata column (still useful for humans, just no longer relied on for uniqueness).

Even after both fixes, 3 articles remain over 3000 chars: the `Article 2` "definitions" article in each of `ssl`/`labor96` (worklaw's own Article 2 is `labor96`; SCL's is `ssl`, both languages) - these are single long glossaries (`Term:\nDefinition\n...`) with no lettered sub-clauses to split on. Left as single oversized chunks for now; would need a different splitter (e.g. one term-definition pair per chunk) if that turns out to matter for retrieval.

In [ ]:
CLAUSE = {
    "ar": re.compile(r"^\s*([\u0623-\u064a\u0640]{1,3})\s*[-\u2013]\s*", re.MULTILINE),
    "en": re.compile(r"^\s*([a-z])\s*[.\-\u2013)]\s*", re.MULTILINE),
}
MAX_CHARS = 3000

def split_article(text, lang, header):
    """Split on top-of-line lettered clause markers. Returns [(clause_letter_or_None, chunk_text), ...];
    a single (None, text) if fewer than 2 markers are found (nothing sensible to split on)."""
    marks = list(CLAUSE[lang].finditer(text))
    if len(marks) < 2:
        return [(None, text)]
    out = []
    for i, m in enumerate(marks):
        end = marks[i + 1].start() if i + 1 < len(marks) else len(text)
        out.append((m.group(1), f"{header}\n{text[m.start():end].strip()}"))
    return out

rows = []
unsplit_long = []  # articles over MAX_CHARS that still ended up as one chunk (no usable clause markers)
for fname, (law, lang) in FILES.items():
    df = pd.read_excel(KB / fname)
    for _, row in df.iterrows():
        art = str(row["Article #"])
        text = str(row["Content"]).strip()
        header = text.split("\n")[0]
        is_long = len(text) > MAX_CHARS
        pieces = split_article(text, lang, header) if is_long else [(None, text)]
        if is_long and len(pieces) == 1:
            unsplit_long.append((law, lang, art, len(text)))
        for i, (clause, body) in enumerate(pieces):
            rows.append({
                "chunk_id":       f"{law}_{lang}_{art}" + (f"_{i}" if clause else ""),
                "law":            law,
                "language":       lang,
                "article_number": art,
                "clause":         clause,
                "text":           body,
                "source_file":    fname,
            })

chunks = pd.DataFrame(rows)
assert chunks["chunk_id"].is_unique, "chunk_id collision - a source article/law/lang combo isn't as unique as assumed"
chunks.to_parquet(KB.parent / "chunks.parquet", index=False)

print(chunks.shape)
print(chunks.groupby(["law", "language"]).size())
L = chunks["text"].str.len()
print(f"med={int(L.median())} p90={int(L.quantile(.9))} max={L.max()} over3k={int((L > 3000).sum())}")
if unsplit_long:
    print("still-unsplit long articles (no usable clause markers):")
    for x in unsplit_long:
        print(" ", x)